# Lensed point sources & time-delay cosmography

A point source behind an EPL + shear deflector, fit to **image positions, relative
fluxes, and relative time delays** — not a pixel grid — with a cosmology whose only
free parameter is `H0` (the classic time-delay `H0` inference).

:::{admonition} Advanced / external dependencies
:class: warning
Unlike the imaging tutorials there is no `SimulatorConfig`/`ImageData`. This notebook
also uses **lenstronomy** to set up the quad and cross-check the physics. It samples
with `gigalens_research`'s MCLMC; the in-repo equivalent is
`gigalens.jax.experimental.mclmc.MCLMC_JIT` (same signature — see the
*Run MCLMC* how-to).
:::

In [ ]:
# ---- environment ----
%matplotlib inline
import os
os.environ['JAX_PLATFORMS'] = 'cpu'   # tiny problem; remove to use a free GPU
import sys
# vendored diagnostics helper lives next to this notebook (demos/)
sys.path.insert(0, os.path.dirname(os.path.abspath('mclmc_diagnostics.py')))

import numpy as np
import jax, jax.numpy as jnp
import optax
import tensorflow_probability.substrates.jax as tfp
import matplotlib as mpl
import matplotlib.pyplot as plt
from corner import corner
from scipy.stats import norm
tfd = tfp.distributions

from gigalens.jax import point_source as ps
from gigalens.jax.point_source import PointSourceData
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_prob_model import ProbModel
from gigalens.jax.inference import ModellingSequence
from gigalens.jax.cosmo import wCDM_Cosmo
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.light.point_source import PointSource
from gigalens_research.inference.mclmc import MCLMC_JIT
from mclmc_diagnostics import plot_mclmc_diagnostics

from lenstronomy.LensModel.lens_model import LensModel as LtLensModel
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.Plots import lens_plot
print('jax', jax.__version__, '| devices', jax.devices())

## Part A — physics port vs lenstronomy (identity check)

The scene point-source physics primitives (`_total_deflection`, `_magnification` = det A via
forward-mode AD, `_fermat_potential`) match lenstronomy on real image positions of an EPL+shear
quad. This is the same Link-A check the validation suite runs (`tests/validation/test_point_source.py`).

*Gates:* |Δ Fermat| ≤ 1e-4 arcsec²; det A rel ≤ 1e-4 (float32 in this container env; the
validated tolerances).

In [ ]:
epl_c = dict(theta_E=1.2, gamma=2.05, e1=0.05, e2=-0.03, center_x=0.01, center_y=-0.02)
shr_c = dict(gamma1=0.03, gamma2=0.02)
lm = LtLensModel(['EPL', 'SHEAR'])
kw_c = [dict(**epl_c), dict(**shr_c, ra_0=0, dec_0=0)]
xs, ys = LensEquationSolver(lm).image_position_from_source(
    0.0, 0.0, kw_c, min_distance=0.01, search_window=6, precision_limit=1e-10)

# scene physics read a list of mass profiles + a {0: params, 1: params} dict (aligned to the
# lens plane's mass Components), exactly as PointSourceLikelihoodTerm calls them internally.
profs = [EPL(niter=18), Shear()]
mp_c = {0: {k: jnp.asarray(v) for k, v in epl_c.items()},
        1: {k: jnp.asarray(v) for k, v in shr_c.items()}}
X, Y = jnp.asarray(xs), jnp.asarray(ys)
ax, ay = ps._total_deflection(profs, mp_c, X, Y)
sx, sy = float(jnp.mean(X - ax)), float(jnp.mean(Y - ay))            # source = mean delensed
fp_gl = np.asarray(ps._fermat_potential(profs, mp_c, X, Y, jnp.asarray(sx), jnp.asarray(sy)))
fp_ls = np.asarray(lm.fermat_potential(np.asarray(xs), np.asarray(ys), kw_c,
                                       x_source=sx, y_source=sy)).reshape(-1)
detA_gl = np.asarray(ps._magnification(profs, mp_c, X, Y)).reshape(-1)
detA_ls = 1.0 / np.asarray(lm.magnification(np.asarray(xs), np.asarray(ys), kw_c)).reshape(-1)
print('images found:', len(xs))
print('Fermat   max |diff|   :', np.abs(fp_gl - fp_ls).max())
print('det A    max rel err  :', (np.abs(detA_gl - detA_ls) / np.abs(detA_ls)).max())

## Part B — a simulated system and its (self-consistent) observed data

Truth mass model + one source → image positions (lenstronomy solver, exact), then the observed
quantities the loss uses: inverse-magnification fluxes `(1/mu)^2` and relative time delays,
generated from the **same** forward model so the loss is ~0 at truth. Images are sorted by
arrival time. `amp` (intrinsic-flux nuisance, prior N(1, 0.1)) is 1.0 at truth.

In [ ]:
# ---- TRUTH ----
epl_t = dict(theta_E=1.167, gamma=2.0, e1=0.049, e2=0.083, center_x=0.005, center_y=0.011)
shr_t = dict(gamma1=0.078, gamma2=0.015)
amp_t, H0_t = 1.0, 70.0
z_lens, z_source, Om0 = 0.2262, 0.3544, 0.3    # SN-Zwicky-like redshifts
src_x_t, src_y_t = 0.0, 0.006

kw_t = [dict(**epl_t), dict(**shr_t, ra_0=0, dec_0=0)]
cx, cy = LensEquationSolver(lm).image_position_from_source(
    src_x_t, src_y_t, kw_t, min_distance=0.01, search_window=6, precision_limit=1e-10)
xs, ys = np.asarray(cx[:4]), np.asarray(cy[:4])
assert len(xs) == 4, 'expected a quad; adjust the source position'

mp_t = {0: {k: jnp.float32(v) for k, v in epl_t.items()},
        1: {k: jnp.float32(v) for k, v in shr_t.items()}}
# Fixed flat-LambdaCDM comoving integrals (Om0 fixed) — the same path the term selects when
# only H0 is free, so the fit is self-consistent (truth recovers exactly).
cl, cs = ps.precompute_comoving_distances(z_lens, z_source, Om0)

def tdall(x, y):
    f = ps._fermat_potential(profs, mp_t, x, y, jnp.float32(src_x_t), jnp.float32(src_y_t))
    return ps._time_delay_days_fixed(f, jnp.float32(H0_t), cl, cs, z_lens, z_source)

X0, Y0 = jnp.asarray(xs, jnp.float32), jnp.asarray(ys, jnp.float32)
order = np.argsort(np.asarray(tdall(X0, Y0)))      # earliest arrival first
xs_s, ys_s = xs[order], ys[order]
X = jnp.asarray(xs_s, jnp.float32)
Y = jnp.asarray(ys_s, jnp.float32)
detA_t = np.asarray(ps._magnification(profs, mp_t, X, Y)).reshape(-1)
flux_obs = (detA_t / amp_t) ** 2
tds = np.asarray(tdall(X, Y)).reshape(-1)
td_obs = tds - tds[0]
# derived truth source (mean of delensed image positions) — used only for plots
axt, ayt = ps._total_deflection(profs, mp_t, X, Y)
sx_t, sy_t = float(jnp.mean(X - axt)), float(jnp.mean(Y - ayt))
print('sorted image x :', np.round(xs_s, 4))
print('sorted image y :', np.round(ys_s, 4))
print('flux_obs (1/mu)^2:', np.round(flux_obs, 5))
print('td_obs (days)  :', np.round(td_obs, 4))

### B.1 — caustics + image positions (truth)

The truth lens's caustics/critical curves (lenstronomy) with the solved image positions (cyan)
and the source (yellow).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
kwargs_caustics = {'color_crit': 'red', 'color_caustic': 'green'}
lens_plot.lens_model_plot(ax, lm, kw_t, numPix=200, deltaPix=0.03,
                          sourcePos_x=sx_t, sourcePos_y=sy_t, point_source=False,
                          with_caustics=True, fast_caustic=True, coord_inverse=False,
                          kwargs_caustics=kwargs_caustics)
ax.plot(xs_s, ys_s, 'cd', ms=11, alpha=0.9, label='images')
ax.plot(sx_t, sy_t, 'y*', ms=16, alpha=0.9, label='source')
for i, (xi, yi) in enumerate(zip(xs_s, ys_s)):
    ax.annotate(str(i), (xi, yi), textcoords='offset points', xytext=(6, 6), color='w', fontsize=12)
ax.legend(loc='upper right'); ax.set_title('Truth: caustics + image positions'); plt.show()

### B.2 — source-plane delensing check

Delens the image positions through the truth model; a good model maps all images back to
(nearly) one source point.

In [ ]:
axd, ayd = ps._total_deflection(profs, mp_t, X, Y)
dx, dy = np.asarray(X - axd).reshape(-1), np.asarray(Y - ayd).reshape(-1)
plt.figure(figsize=(5, 5))
plt.plot(dx, dy, 'D', color='navy', ms=6, label='delensed images')
plt.plot(sx_t, sy_t, '*', color='k', ms=14, label='source (mean)')
plt.title('Source plane (truth)'); plt.xlabel('x (arcsec)'); plt.ylabel('y (arcsec)')
plt.legend(); plt.axis('equal'); plt.show()
print('delensed-position scatter (arcsec): x %.2e  y %.2e' % (dx.std(), dy.std()))

## Part C — prior + scene `ProbModel`

The point source is `Component(PointSource(use_lstsq=False), dict(amp=...))`; its
observation is a `PointSourceData` of image positions, relative fluxes, and relative
time delays (with weights balancing the three terms). The cosmology `wCDM_Cosmo`
samples only `H0`. The **source position is not a free parameter** — it's recovered as
the mean of the delensed image positions.

In [ ]:
ps_comp = Component(PointSource(use_lstsq=False), dict(amp=tfd.Normal(1.0, 0.1)))
epl_comp = Component(EPL(niter=18), dict(
    theta_E=tfd.Uniform(0.5, 2.0), gamma=tfd.TruncatedNormal(2.0, 0.25, 1.5, 2.5),
    e1=tfd.Normal(0.0, 0.1), e2=tfd.Normal(0.0, 0.1),
    center_x=tfd.Normal(0.0, 0.1), center_y=tfd.Normal(0.0, 0.1)))
shear_comp = Component(Shear(), dict(gamma1=tfd.Normal(0.0, 0.1), gamma2=tfd.Normal(0.0, 0.1)))
cosmo_comp = Component(wCDM_Cosmo(z_lens=z_lens, z_source_ref=z_source),
                       dict(H0=tfd.Uniform(0.0, 150.0), Om0=Om0, k=0.0, w0=-1.0))
model = LensModel([Plane(redshift=z_lens, mass=[epl_comp, shear_comp]),
                   Plane(redshift=z_source, light=[ps_comp])], cosmo=cosmo_comp)

# >>> hand-tuned weights (experiment here) <<<
weight_dist, weight_flux, weight_time_delay = 3e3, 5e9, 3e3

data = PointSourceData(ps_comp, np.asarray(X).reshape(-1), np.asarray(Y).reshape(-1),
                       flux_obs, td_obs, weight_dist, weight_flux, weight_time_delay)
prob = ProbModel(model, data)
seq = ModellingSequence(prob)

# z-column names come from the model (never reconstruct them by hand); map each to a label + truth.
names = list(model.z_param_names)
label_by_path = {'cosmo/H0': r'$H_0$',
    'planes/0/mass/0/theta_E': r'$\theta_E$', 'planes/0/mass/0/gamma': r'$\gamma$',
    'planes/0/mass/0/e1': r'$e_1$', 'planes/0/mass/0/e2': r'$e_2$',
    'planes/0/mass/0/center_x': r'$x_{lens}$', 'planes/0/mass/0/center_y': r'$y_{lens}$',
    'planes/0/mass/1/gamma1': r'$\gamma_1$', 'planes/0/mass/1/gamma2': r'$\gamma_2$',
    'planes/1/light/0/amp': r'$Amp$'}
truth_by_path = {'cosmo/H0': H0_t,
    'planes/0/mass/0/theta_E': epl_t['theta_E'], 'planes/0/mass/0/gamma': epl_t['gamma'],
    'planes/0/mass/0/e1': epl_t['e1'], 'planes/0/mass/0/e2': epl_t['e2'],
    'planes/0/mass/0/center_x': epl_t['center_x'], 'planes/0/mass/0/center_y': epl_t['center_y'],
    'planes/0/mass/1/gamma1': shr_t['gamma1'], 'planes/0/mass/1/gamma2': shr_t['gamma2'],
    'planes/1/light/0/amp': amp_t}
labels = [label_by_path[n] for n in names]
truth = np.array([truth_by_path[n] for n in names])

def to_constrained(z_rows):
    # (n, 10) unconstrained z -> (n, 10) constrained array in `names` order.
    xb = model.bijector.forward(jnp.asarray(z_rows).reshape(-1, len(names)))
    return np.stack([np.asarray(xb[n]).reshape(-1) for n in names], axis=1)

def mass_amp_h0(arr):
    # (n, 10) constrained -> ({0: EPL, 1: shear} batched, amp, H0) for the physics functions.
    a = jnp.asarray(arr); col = {n: a[:, i] for i, n in enumerate(names)}
    mp = {0: dict(theta_E=col['planes/0/mass/0/theta_E'], gamma=col['planes/0/mass/0/gamma'],
                  e1=col['planes/0/mass/0/e1'], e2=col['planes/0/mass/0/e2'],
                  center_x=col['planes/0/mass/0/center_x'], center_y=col['planes/0/mass/0/center_y']),
          1: dict(gamma1=col['planes/0/mass/1/gamma1'], gamma2=col['planes/0/mass/1/gamma2'])}
    return mp, col['planes/1/light/0/amp'], col['cosmo/H0']

print('free parameters:', model.num_free_params)
print('z-column order :', names)

## Part D — MAP (initializer)

MAP over the stiff loss is multimodal; used only to seed SVI. Gradient clipping keeps it from
diverging. Expect it near truth on good seeds, and in a secondary mode (mass-slope↔H0 /
ellipticity–shear degeneracy) on others — hence the large `n_samples` and `output_type='best'`.

In [ ]:
map_opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(3e-3))
best_z, best_lp, _ = seq.MAP(map_opt, n_samples=2000, num_steps=1000, seed=2, output_type='best')
best_z = np.asarray(jax.device_get(best_z))   # strip Explicit sharding (JAX 0.10) for MCLMC later
map_vals = to_constrained(best_z)[0]
print('MAP best log-post: %.4g\n' % float(best_lp))
print('%-10s %10s %10s' % ('param', 'truth', 'MAP'))
for lab, t, v in zip(labels, truth, map_vals):
    print('%-10s %10.4f %10.4f' % (lab, t, v))

## Part E — SVI surrogate (run off MAP)

Fits a multivariate-normal surrogate `q_z` in unconstrained space. It initializes MCLMC
(positions + mass matrix) and is one of the corner-plot layers.

In [ ]:
svi_opt = optax.chain(optax.scale_by_adam(), optax.scale_by_schedule(
    optax.polynomial_schedule(init_value=-1e-6, end_value=-3e-3, power=2.0, transition_steps=2000)))
qz_svi, svi_losses = seq.SVI(best_z, svi_opt, n_vi=1000, init_scales=1e-3, num_steps=2000, seed=2)
svi_losses = np.asarray(svi_losses).reshape(-1)
print('SVI final -ELBO:', float(svi_losses[-1]))

plt.figure(figsize=(6, 3))
plt.plot(svi_losses); plt.xlabel('step'); plt.ylabel('-ELBO'); plt.title('SVI loss'); plt.show()

svi_z = np.asarray(jax.device_get(qz_svi.sample(4000, seed=jax.random.PRNGKey(1))))
svi_samples = to_constrained(svi_z)

## Part F — sampling with MCLMC  ← **the part to play with**

`MCLMC_JIT(seq, qz, ...)` takes the `ModellingSequence` and an SVI/MAP surrogate `qz`
and returns unconstrained draws. This notebook imports it from `gigalens_research`;
the in-repo `gigalens.jax.experimental.mclmc.MCLMC_JIT` has the same signature.

In [ ]:
n_chains = 8
num_burnin_steps, num_results = 1000, 2000
frac_tune1, frac_tune2, frac_tune3 = 0.2, 0.6, 0.2

qz = tfd.MultivariateNormalFullCovariance(
    loc=np.asarray(jax.device_get(qz_svi.mean())),
    covariance_matrix=np.asarray(jax.device_get(qz_svi.covariance())))
hist = MCLMC_JIT(seq, qz, n_hmc=n_chains,
                 num_burnin_steps=num_burnin_steps, num_results=num_results,
                 frac_tune1=frac_tune1, frac_tune2=frac_tune2, frac_tune3=frac_tune3,
                 progress_bar=False, seed=2, debug_output=True)
samples = np.asarray(hist.position)[:, -num_results:, :]   # (chains, draws, 10) unconstrained
print('samples shape (chains, draws, params):', samples.shape)

In [ ]:
# ---- convergence + recovery diagnostics ----
n_ch, n_dr = samples.shape[0], samples.shape[1]
post = to_constrained(samples.reshape(-1, len(names))).reshape(n_ch, n_dr, len(names))
post_cd = post.transpose(1, 0, 2)
rhat = np.asarray(tfp.mcmc.potential_scale_reduction(post_cd, independent_chain_ndims=1))
ess = np.asarray(tfp.mcmc.effective_sample_size(post_cd))
ess = np.nansum(ess, axis=0) if ess.ndim > 1 else ess
mclmc_flat = post.reshape(-1, len(names))
mean, std = mclmc_flat.mean(0), mclmc_flat.std(0)
q025, q975 = np.percentile(mclmc_flat, 2.5, 0), np.percentile(mclmc_flat, 97.5, 0)
print('%-10s %9s %9s %9s %7s %9s %7s cov95' % ('param', 'truth', 'mean', 'std', 'z', 'R-hat', 'ESS'))
nc = 0
for i, lab in enumerate(labels):
    z = (mean[i] - truth[i]) / std[i] if std[i] > 0 else np.nan
    c = q025[i] <= truth[i] <= q975[i]; nc += int(c)
    print('%-10s %9.4f %9.4f %9.4f %7.2f %9.2g %7.1f  %s' % (lab, truth[i], mean[i], std[i], z, rhat[i], ess[i], 'Y' if c else 'N'))
print('\ncoverage: %d/10   |   max R-hat: %.3g   (converged iff all R-hat < 1.01)' % (nc, np.nanmax(rhat)))
if np.nanmax(rhat) > 1.01:
    print('>>> R-hat above 1.01: not fully converged; increase burn-in/results (ESS shows if chains move).')

## Part G — MCLMC tuning diagnostics

Five panels vs. step: chain-wise step size, chain-wise `L`, inverse-mass-matrix eigenvalue
spread, per-step energy-error ratio `xi`, and a finite/NaN success heatmap. Dashed lines mark the
three tuning-stage boundaries. A step size collapsing toward ~0 during tuning is the
adaptation-collapse signature of non-mixing on a too-stiff loss.

In [ ]:
fig = plot_mclmc_diagnostics(hist, num_burnin_steps=num_burnin_steps,
                             frac_tune1=frac_tune1, frac_tune2=frac_tune2, frac_tune3=frac_tune3,
                             chain=0)
fig.suptitle('MCLMC tuning diagnostics', y=1.005)
plt.show()

## Part H — chains + marginals for $\theta_E$, $\gamma$, $H_0$

Per-parameter trace (left) + per-chain fitted-normal marginal (right).

In [ ]:
show = [('planes/0/mass/0/theta_E', r'$\theta_E$'),
        ('planes/0/mass/0/gamma', r'$\gamma$'),
        ('cosmo/H0', r'$H_0$')]
for path, ttl in show:
    i = names.index(path)
    fig = plt.figure(figsize=(10, 3))
    gs = plt.GridSpec(1, 2, width_ratios=[1, .4])
    ax1 = plt.subplot(gs[0, 0]); ax1.grid(True)
    for c in range(n_ch):
        ax1.plot(post[c, :, i], alpha=0.6, lw=0.8)
    ax1.axhline(truth[i], color='k', ls='--', lw=1.5)
    ax1.set_title('MCLMC ' + ttl); ax1.set_xlabel('Iterations')
    ax2 = plt.subplot(gs[0, 1], sharey=ax1); ax2.grid(True)
    lo, hi = post[:, :, i].min(), post[:, :, i].max()
    bins = np.linspace(lo, hi, 60)
    for c in range(n_ch):
        mu, sigma = norm.fit(post[c, :, i])
        ax2.plot(norm.pdf(bins, mu, sigma), bins, lw=2, alpha=0.8)
    ax2.axhline(truth[i], color='k', ls='--', lw=1.5)
    ax2.tick_params(axis='y', labelleft=False); ax2.set_xticks([])
    plt.tight_layout(); plt.show()

## Part I — corner plot: MCLMC posterior + truth

The MCLMC posterior with **truth as a green crosshair** and median±(16/84%) titles.

In [ ]:
navy = [(0.0, 0.0, 0.5, 0.0), (0.0, 0.0, 0.5, 0.4), (0.0, 0.0, 0.5, 0.6),
        (0.0, 0.0, 0.5, 0.8), (0.0, 0.0, 0.5, 1.0)]
fig = corner(mclmc_flat, labels=labels, truths=truth, truth_color='#006400',
             show_titles=True, title_fmt='.3f', plot_contours=True, fill_contours=True,
             contourf_kwargs={'colors': navy},
             label_kwargs={'fontsize': 16}, title_kwargs={'fontsize': 11})
for ax in fig.get_axes():
    ax.tick_params(axis='both', which='major', labelsize=10)
plt.show()

### I.1 — corner overlay: SVI surrogate vs MCLMC (with MAP star)

SVI surrogate (blue) vs MCLMC posterior (orange), truth crosshair (black), MAP point (red star).

In [ ]:
lo = np.minimum.reduce([np.percentile(mclmc_flat, 0.5, 0), truth, map_vals])
hi = np.maximum.reduce([np.percentile(mclmc_flat, 99.5, 0), truth, map_vals])
span = np.where(hi > lo, hi - lo, np.abs(hi) + 1e-6)
rng = list(zip(lo - 0.2 * span, hi + 0.2 * span))
ck = dict(labels=labels, range=rng, plot_datapoints=False, plot_density=True,
          hist_kwargs=dict(density=True), smooth=1.0)
fig = corner(svi_samples, color='C0', **ck)
corner(mclmc_flat, fig=fig, color='C1', **ck)
from corner import overplot_lines, overplot_points
overplot_lines(fig, truth, color='k', ls='--', lw=1.2)
overplot_points(fig, truth[None], marker='s', color='k', ms=5)
overplot_points(fig, map_vals[None], marker='*', color='C3', ms=14)
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], color='C0', label='SVI surrogate'),
                    Line2D([], [], color='C1', label='MCLMC'),
                    Line2D([], [], color='k', ls='--', label='truth'),
                    Line2D([], [], color='C3', marker='*', ls='', label='MAP')],
           loc='upper right', fontsize=11, frameon=False)
plt.show()

## Part J — best-fit predictions: positions, magnifications, time delays

Best-fit = MCLMC posterior median. Solve the lens equation at the best-fit mass model for the
modeled image positions, and compare per-image **magnification** and **relative time delay**
(simulated truth vs predicted ± posterior std) on the caustic diagram.

In [ ]:
best_fit = np.median(mclmc_flat, axis=0)              # constrained median (10,)
mp_best, amp_best, H0_best = mass_amp_h0(best_fit[None])   # single best-fit (batch 1)
mp_all, amp_all, H0_all = mass_amp_h0(mclmc_flat)         # batched over all draws

# best-fit mass kwargs for lenstronomy
kw_bf = [dict(theta_E=best_fit[names.index('planes/0/mass/0/theta_E')],
              gamma=best_fit[names.index('planes/0/mass/0/gamma')],
              e1=best_fit[names.index('planes/0/mass/0/e1')],
              e2=best_fit[names.index('planes/0/mass/0/e2')],
              center_x=best_fit[names.index('planes/0/mass/0/center_x')],
              center_y=best_fit[names.index('planes/0/mass/0/center_y')]),
         dict(gamma1=best_fit[names.index('planes/0/mass/1/gamma1')],
              gamma2=best_fit[names.index('planes/0/mass/1/gamma2')], ra_0=0, dec_0=0)]

# modeled source (mean delensed under best-fit) and modeled image positions
ax_bf, ay_bf = ps._total_deflection(profs, mp_best, X, Y)
src_x_bf = float(np.mean(np.asarray(X - ax_bf)))
src_y_bf = float(np.mean(np.asarray(Y - ay_bf)))
xm, ym = LensEquationSolver(lm).image_position_from_source(
    src_x_bf, src_y_bf, kw_bf, min_distance=0.01, search_window=6, precision_limit=1e-10)

# per-image predicted magnification & relative TD (best-fit) + posterior-std uncertainties
detA_bf = np.asarray(ps._magnification(profs, mp_best, X, Y)).reshape(-1)
mag_pred = np.abs(1.0 / detA_bf)
def _td_days(mp_b, H0_b):
    f = ps._fermat_potential(profs, mp_b, X.reshape(-1, 1), Y.reshape(-1, 1),
                             jnp.asarray(src_x_bf), jnp.asarray(src_y_bf))
    return np.asarray(ps._time_delay_days_fixed(f, H0_b, cl, cs, z_lens, z_source))
TD_bf = _td_days(mp_best, H0_best).reshape(-1); TD_bf = TD_bf - TD_bf[0]

detA_s = np.asarray(ps._magnification(profs, mp_all, X, Y))       # (n_images, n_draws)
mag_unc = np.std(np.abs(1.0 / detA_s), axis=1)
axs_s, ays_s = ps._total_deflection(profs, mp_all, X.reshape(-1, 1), Y.reshape(-1, 1))
src_sx = np.mean(np.asarray(X.reshape(-1, 1) - axs_s), axis=0)    # (n_draws,)
src_sy = np.mean(np.asarray(Y.reshape(-1, 1) - ays_s), axis=0)
f_s = ps._fermat_potential(profs, mp_all, X.reshape(-1, 1), Y.reshape(-1, 1),
                           jnp.asarray(src_sx), jnp.asarray(src_sy))
TD_s = np.asarray(ps._time_delay_days_fixed(f_s, H0_all, cl, cs, z_lens, z_source))
TD_s = TD_s - TD_s[0:1]
TD_unc = np.std(TD_s, axis=1)

mag_true = np.abs(1.0 / detA_t)
fig, ax = plt.subplots(figsize=(9, 8))
lens_plot.lens_model_plot(ax, lm, kw_bf, numPix=200, deltaPix=0.03, point_source=False,
                          with_caustics=True, fast_caustic=True, coord_inverse=False,
                          kwargs_caustics={'color_crit': 'red', 'color_caustic': 'green'})
ax.plot(xs_s, ys_s, 'd', ms=13, color='lightblue', alpha=0.9, label='observed position')
ax.plot(np.asarray(xm), np.asarray(ym), 'd', ms=8, color='navy', alpha=0.9, label='predicted position')
ax.plot(src_x_bf, src_y_bf, '*', ms=13, color='yellow', label='predicted source')
for i in range(len(xs_s)):
    s = ('Sim Mag: %.3f\nPred Mag: %.3f$\\pm$%.3f\nSim TD: %.3f\nPred TD: %.3f$\\pm$%.3f'
         % (mag_true[i], mag_pred[i], mag_unc[i], td_obs[i], TD_bf[i], TD_unc[i]))
    ax.annotate(s, (xs_s[i], ys_s[i]), textcoords='offset points', xytext=(8, -8),
                fontsize=8, color='white')
ax.legend(loc='upper right', facecolor='lightgray'); ax.set_title('Best-fit predictions vs truth')
plt.show()

## Part K — residual image and reduced $\chi^2$

Renders an "observed" image (truth) and a "best-fit model" image with lenstronomy's
`LENSED_POSITION` point-source model and a Gaussian PSF, then reports the reduced chi-squared.
Point-source amplitudes are `amp * |mu|` per image. (The point-source *fit* uses positions /
fluxes / time-delays, not pixels — this image is a visualization only.)

In [ ]:
from lenstronomy.PointSource.point_source import PointSource as LenstroPS
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.Data.imaging_data import ImageData as LtImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.Util import image_util
from mpl_toolkits.axes_grid1 import make_axes_locatable

deltaPix, numPix = 0.05, 120
background_rms, exp_time = 0.01, 1000.0

def gaussian_kernel(n=25, fwhm_arcsec=0.12, deltaPix=deltaPix):
    fwhm_pix = fwhm_arcsec / deltaPix
    sig = fwhm_pix / 2.3548
    a = np.arange(n) - n // 2
    xx, yy = np.meshgrid(a, a)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sig ** 2))
    return (k / k.sum()).astype(np.float64)

kernel = gaussian_kernel()

def render(mass_kwargs, x_img, y_img, point_amp, add_noise=False):
    lens_model = LtLensModel(['EPL', 'SHEAR'])
    ps_model = LenstroPS(point_source_type_list=['LENSED_POSITION'], lens_model=lens_model,
                         fixed_magnification_list=[False])
    ps_kwargs = [{'ra_image': np.asarray(x_img), 'dec_image': np.asarray(y_img),
                  'point_amp': np.asarray(point_amp)}]
    psf = PSF(psf_type='PIXEL', kernel_point_source=kernel)
    lt_data = LtImageData(background_rms=background_rms, exposure_time=exp_time,
                          ra_at_xy_0=-numPix / 2 * deltaPix, dec_at_xy_0=-numPix / 2 * deltaPix,
                          transform_pix2angle=np.array([[deltaPix, 0.], [0., deltaPix]]),
                          image_data=np.zeros((numPix, numPix)))
    im = ImageModel(data_class=lt_data, psf_class=psf, lens_model_class=lens_model,
                    point_source_class=ps_model)
    img = im.image(kwargs_lens=mass_kwargs, kwargs_ps=ps_kwargs)
    if add_noise:
        img = img + image_util.add_poisson(img, exp_time=exp_time) + image_util.add_background(img, sigma_bkd=background_rms)
    return np.asarray(img)

amp_scale = 10.0
observed_img = render([dict(**epl_t), dict(**shr_t, ra_0=0, dec_0=0)], xs_s, ys_s,
                      amp_scale * amp_t * mag_true, add_noise=True)
model_mag = np.abs(1.0 / np.asarray(ps._magnification(profs, mp_best,
                   jnp.asarray(xm), jnp.asarray(ym))).reshape(-1))
model_img = render(kw_bf, np.asarray(xm), np.asarray(ym), amp_scale * amp_best[0] * model_mag)

err_map = np.sqrt(background_rms ** 2 + np.clip(observed_img, 0, None) / exp_time)
dof = np.sum(observed_img > 0) - len(names)
red_chi2 = np.sum(((observed_img - model_img) / err_map) ** 2) / max(dof, 1)

extent = (-numPix / 2 * deltaPix, numPix / 2 * deltaPix, -numPix / 2 * deltaPix, numPix / 2 * deltaPix)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 6))
for ax, img, ttl in [(ax0, observed_img, 'Simulated image'), (ax1, model_img, 'Best-fit model')]:
    im = ax.imshow(img, extent=extent, cmap='inferno',
                   norm=mpl.colors.PowerNorm(0.65, vmin=0, vmax=max(observed_img.max(), 1e-6)),
                   origin='lower')
    ax.set_title(ttl); ax.set_xticks([]); ax.set_yticks([])
    cax = make_axes_locatable(ax).append_axes('right', size='5%', pad=0.05)
    fig.colorbar(im, cax=cax)
ax1.text(0.03, 0.95, r'$\chi^2/{\rm dof}$ = %.2f' % red_chi2, color='white',
         transform=ax1.transAxes, va='top', fontsize=13)
plt.tight_layout(); plt.show()
print('reduced chi^2 (Gaussian-PSF render):', round(float(red_chi2), 3))

## Notes & known issues

1. **Faithful to the old-API demo.** Same truth system, image positions, observed
   fluxes/time-delays, priors, hand-tuned weights, and sampler (MCLMC-off-SVI). The physics
   (`_magnification`, `_fermat_potential`) is re-verified against lenstronomy in Part A.
2. **Point-source integration is a scene `Dataset`.** `PointSourceData` +
   `PointSourceLikelihoodTerm` compute `L_D + L_F + L_TD`; the source position is derived (mean
   delensed), so 10 parameters are sampled — the source enters only through `amp`.
3. **Cosmology is differentiable** (`wCDM_Cosmo`), here with `Om0`/`w0`/`k` fixed (only `H0`
   free), matching the original. Freeing `Om0`/`w0` infers them with exact gradients.
4. **Hand-tuned weights** make the loss stiff and the posterior widths uncalibrated; softening
   `weight_flux` toward realistic measurement inverse-variances would make widths physical.
5. **Sampler.** MCLMC-off-SVI mixes on the stiff loss where HMC does not; `hist` and Part G's
   diagnostics are MCLMC-specific.